In [13]:
import json
import time
from IPython.display import display, Markdown, HTML
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL)

In [10]:
def build_messages(message_dicts):
    type_map = {
        "system" : SystemMessage,
        "user" : HumanMessage,
        "assistant" : AIMessage
    }
    return [type_map[m["role"]](content=m["content"]) for m in message_dicts]

def chat(messages, **kwargs):
    _llm = ChatOllama(model = MODEL, **kwargs) if kwargs else _llm
    lc_messages = build_messages(messages)
    start = time.time()
    response = llm.invoke(messages)
    elapsed = time.time() - start
    content = response.content
    display(Markdown(content))
    print(f"\n⏱️ Response time {elapsed:.2f}s")
    return content

def show_messages(messages):
    colors = {"system": "#e74c3c", "user": "#3498db", "assistant": "#2ecc71"}
    html = ""

    for msg in messages:
        role = msg["role"]
        color = colors.get(role, "#888")
        content_preview = msg["content"][:300] + ("..." if len(msg["content"]) > 300 else "")
        html += (
            f'<div style="margin:6px 0;padding:8px 12px;border-left:4px solid {color};'
            f'background:#1e1e1e;border-radius:4px;">'
            f'<strong style="color:{color};text-transform:uppercase;">{role}</strong>'
            f'<br><span style="color:#ccc;white-space:pre-wrap;">{content_preview}</span></div>'
        )
    display(HTML(html))

print(f"✅ Using model: {MODEL}")


✅ Using model: qwen2.5:1.5b


In [19]:
# ZERO SHOT

print("ZERO SHORT")

query = "Translate the following English sentence to Spanish:\n\n\"What are you doing?\""

messages = [
    {"role": "user", "content": query}
]
show_messages(messages)
_ = chat(messages, temperature=0.0)

ZERO SHORT


The translation of "What are you doing?" in Spanish is "¿Qué estás haciendo?"


⏱️ Response time 1.09s


In [22]:
# ONE SHOT

one_shot = [
    {"role":"system", "content":"Classify the emotion in each sentence. Reply with exactly one word: Happy, Sad, Angry, or Fearful."},
    {"role":"user", "content": "India won the match"},
    {"role":"assistant", "content": "Happy"},
    {"role":"user", "content": "Pakistan lost the match"}
]
show_messages(one_shot)
_ = chat(one_shot, temperature = 0.0)

Sad


⏱️ Response time 0.41s


In [26]:
few_shot = [
    {"role":"system", "content":"Classify the emotion in each sentence. Reply with exactly one word: Happy, Sad, Angry, or Fearful."},
    {"role":"user", "content": "India won the match"},
    {"role":"assistant", "content": "Happy"},
    {"role":"user", "content": "Pakistan lost the match"},
    {"role":"assistant", "content": "Sad"},
    {"role":"user", "content": "Pakistan fans throw the bottles to the ground, after loosing the match"},
    {"role":"assistant", "content": "Angry"},
    {"role":"user", "content": "Pakistan players afraid to face the ball bowled by Bumrah"} 
]
show_messages(few_shot)
_ = chat(few_shot, temperature = 0.0)

few_shot[-1] = {"role": "user", "content": "Indian fans celebrated their victory"}
print("Next example")
_ = chat(few_shot, temperature = 0.0)

Fearful


⏱️ Response time 0.72s
Next example


Happy


⏱️ Response time 0.40s


In [33]:
# ROLE + FEW SHOT

system = {
    "role": "system", "content": (
    "You are cricket analysis assistant. Be strict and concise."
    "Based on the questions assign, reply with Out or Not-out"
    )
}
messages = [
    {"role": "user", "content": "The ball hits the stumps"},
    {"role": "assistant", "content": "Out"},
    {"role": "user", "content": "The ball pitched in line, strcuk the pad, impact in line, wickets missing."},
    {"role": "assistant", "content": "Not-Out"},
    {"role": "user", "content": "The ball took the edge of the bat and went straight to wicketkeeper, who made a clean catch."},
    {"role": "assistant", "content": "Out"},     
    {"role": "user", "content": "The fielder takes the catch near the boundary."},     
]

show_messages(messages)
_ = chat(messages, temperature = 0.7)

Out


⏱️ Response time 0.83s


In [35]:
# SRUCTURED OUTPUT

input_text = "MS Dhoni was the captain of Indian cricket team. He has won Padma Bhushan in 2018 and Padma Shri in 2019."

messages = [
    {"role": "system", "content": "You are a JSON extractor. Extract the information from the text and respond with valid json only."
    "Keys : name, sport, team, achievements(list of objects with awards and year)"},
    {"role": "user","content": input_text}
]
show_messages(messages)
response = chat(messages, temperature = 0.0)

try:
    json.loads(response)
    print("✅ Valid JSON")
except json.JSONDecodeError:
    print("❌ Invalid JSON")

{
    "name": "MS Dhoni",
    "sport": "Cricket",
    "team": "India",
    "achievements": [
        {
            "awards": ["Padma Bhushan"],
            "year": 2018
        },
        {
            "awards": ["Padma Shri"],
            "year": 2019
        }
    ]
}


⏱️ Response time 10.45s
✅ Valid JSON
